# Projeto Data Major — Etapa 03: Load

**Responsável:** Lucas de Siqueira Cavalcanti


Esta etapa lê o parquet gerado pelo Transform, valida a estrutura, disponibiliza uma view SQL para consultas e persiste uma amostra em SQLite para inspeção via DBeaver.


In [1]:
from google.colab import drive
import os
from pyspark.sql import SparkSession
import sqlite3
import pandas as pd
import glob

## 1. Configuração — Drive e caminhos


In [2]:
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/Topicos_BD'
path_processed = os.path.join(base_path, 'processed', 'sinan_dengue_processed')
db_path   = os.path.join(base_path, 'database', 'dengue_db.db')

os.makedirs(os.path.dirname(db_path), exist_ok=True)

Mounted at /content/drive


## 2. Sessão Spark


In [3]:
spark = SparkSession.builder \
    .appName('DataMajor_Load') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

## 3. Leitura do dataset gerado pelo Transform


In [4]:
df_final = spark.read.parquet(path_processed)

total = df_final.count()
print(f'Registros carregados: {total:,}')
print(f'Quantidade de colunas: {len(df_final.columns)}')

print('\nSchema final do dataset:')
df_final.printSchema()

Registros carregados: 6,901,591
Quantidade de colunas: 36

Schema final do dataset:
root
 |-- HOSPITALIZ: integer (nullable = true)
 |-- NU_IDADE_N: string (nullable = true)
 |-- ID_MUNICIP: string (nullable = true)
 |-- ID_OCUPA_N: string (nullable = true)
 |-- NU_ANO: integer (nullable = true)
 |-- FEBRE: integer (nullable = true)
 |-- MIALGIA: integer (nullable = true)
 |-- CEFALEIA: integer (nullable = true)
 |-- VOMITO: integer (nullable = true)
 |-- NAUSEA: integer (nullable = true)
 |-- DOR_RETRO: integer (nullable = true)
 |-- DOR_COSTAS: integer (nullable = true)
 |-- EXANTEMA: integer (nullable = true)
 |-- LEUCOPENIA: integer (nullable = true)
 |-- DIABETES: integer (nullable = true)
 |-- HIPERTENSA: integer (nullable = true)
 |-- RENAL: integer (nullable = true)
 |-- HEPATOPAT: integer (nullable = true)
 |-- HEMATOLOG: integer (nullable = true)
 |-- idade_anos: integer (nullable = true)
 |-- nome_municipio: string (nullable = true)
 |-- uf_sigla: string (nullable = true)
 |

## 4. View SQL para consultas


In [5]:
df_final.createOrReplaceTempView("tb_dengue_final")

In [6]:
# Validação simples da view SQL
spark.sql("SELECT * FROM tb_dengue_final LIMIT 5").show()

+----------+----------+----------+----------+------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+--------------+--------+-------+------+----------+---------+--------+---------------+---------+--------+----------+--------+-----------+---------------+----------------+-----------------------+
|HOSPITALIZ|NU_IDADE_N|ID_MUNICIP|ID_OCUPA_N|NU_ANO|FEBRE|MIALGIA|CEFALEIA|VOMITO|NAUSEA|DOR_RETRO|DOR_COSTAS|EXANTEMA|LEUCOPENIA|DIABETES|HIPERTENSA|RENAL|HEPATOPAT|HEMATOLOG|idade_anos|nome_municipio|uf_sigla|uf_nome|regiao|   _row_id|is_mulher|is_homem|is_sexo_ausente|is_branco|is_preto|is_amarelo|is_pardo|is_indigena|is_raca_ausente|escolaridade_ord|is_escolaridade_ausente|
+----------+----------+----------+----------+------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+--------------+--------+-------+------+----------+-

In [7]:
# Tamanho total dos arquivos parquet do Transform
padrao_busca = os.path.join(path_processed, '**', '*.parquet')
arquivos_parquet = glob.glob(padrao_busca, recursive=True)

tamanho_mb = sum(os.path.getsize(f) for f in arquivos_parquet) / 1e6

print(f"Tamanho total: {tamanho_mb:.2f} MB")

Tamanho total: 60.60 MB


## 5. Persistência de amostra em SQLite


In [8]:
# SQLite não é adequado para milhões de registros — utiliza-se uma
# amostra apenas para demonstração da carga. O Data Lake permanece
# sendo o parquet completo lido pelo Spark.
conn = sqlite3.connect(db_path)

df_final_reloaded = spark.read.parquet(path_processed)
df_amostra = df_final_reloaded.limit(100000).toPandas()
df_amostra.to_sql('sinan_dengue', conn, if_exists='replace', index=False)

conn.close()

print(f'Amostra de {len(df_amostra):,} registros persistida em: {db_path}')

Amostra de 100,000 registros persistida em: /content/drive/MyDrive/Topicos_BD/database/dengue_db.db


In [9]:
# spark.stop()